# Multilingual RAG Evaluation with LangChain and RAGAS

**Author**: Dr. Rajan Prasad Tripathi | AUT AI Innovation Lab

This cookbook demonstrates how to evaluate Retrieval-Augmented Generation (RAG) systems across multiple languages using LangChain and RAGAS. We cover:

1. Setting up a multilingual RAG pipeline with LangChain
2. Generating multilingual test datasets with RAGAS
3. Evaluating RAG performance across English, Chinese, Arabic, and Uzbek
4. Analyzing cross-lingual retrieval quality and faithfulness

## Why Multilingual RAG Evaluation Matters

Most RAG evaluation frameworks assume English-only contexts. However, real-world applications often need to:
- Retrieve documents in one language and answer in another
- Handle low-resource languages with limited training data
- Maintain consistent quality across diverse linguistic contexts

This notebook addresses these challenges with practical implementation patterns.

## Setup and Dependencies

In [ ]:
# Install required packages
!pip install -q langchain langchain-openai langchain-community ragas chromadb tiktoken pypdf

# Optional: For multilingual embeddings
!pip install -q sentence-transformers langchain-huggingface

In [ ]:
import os
import asyncio
from typing import List, Dict, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# LangChain imports
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# RAGAS imports
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
    answer_similarity,
    answer_correctness
)
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset
from ragas.testset import TestsetGenerator
from ragas.testset.synthesizers.single_hop import SingleHopSpecificQuerySynthesizer
from ragas.testset.transforms.extractors import NERExtractor, SummaryExtractor
from ragas.testset.transforms.splitters import HeadlineSplitter

# Dataset imports
from datasets import Dataset

print("All imports successful!")

In [ ]:
# Set your OpenAI API key
os.environ["OPENAI_API_KEY"] = "your-openai-api-key-here"

# Initialize LLM and embeddings
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# Use multilingual embeddings (better for non-English)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)

print("LLM and embeddings initialized!")

## Part 1: Build a Multilingual RAG Pipeline

In [ ]:
def create_multilingual_rag_pipeline(
    documents_path: str,
    chunk_size: int = 1000,
    chunk_overlap: int = 200
) -> Tuple[Chroma, Callable]:
    """
    Create a multilingual RAG pipeline with Chroma vector store.
    
    Args:
        documents_path: Path to documents (PDF or text files)
        chunk_size: Size of text chunks
        chunk_overlap: Overlap between chunks
    
    Returns:
        Tuple of (vectorstore, retrieval_chain)
    """
    # Load documents
    if documents_path.endswith('.pdf'):
        loader = PyPDFLoader(documents_path)
    else:
        loader = TextLoader(documents_path, encoding='utf-8')
    
    docs = loader.load()
    
    # Split documents with multilingual-aware splitting
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", "。", "", "\\.", " ", ""]  # Include Chinese period
    )
    splits = text_splitter.split_documents(docs)
    
    # Create vector store
    vectorstore = Chroma.from_documents(
        documents=splits,
        embedding=embeddings,
        persist_directory="./chroma_db"
    )
    
    # Create RAG chain with multilingual prompt
    template = """You are a helpful assistant that answers questions based on the provided context.
Answer in the same language as the question. Be accurate and faithful to the context.

Context:
{context}

Question: {question}

Answer:"""
    
    prompt = ChatPromptTemplate.from_template(template)
    
    def retrieve_and_generate(question: str) -> str:
        retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
        docs = retriever.invoke(question)
        context = "\n\n".join(doc.page_content for doc in docs)
        chain = prompt | llm | StrOutputParser()
        return chain.invoke({"context": context, "question": question})
    
    return vectorstore, retrieve_and_generate

print("RAG pipeline function defined!")

## Part 2: Generate Multilingual Test Datasets

The key challenge in multilingual RAG evaluation is generating high-quality test data in non-English languages. RAGAS provides `adapt_prompts()` for this purpose.

In [ ]:
async def generate_multilingual_testset(
    documents: List,
    llm,
    embedding_model,
    language: str = "chinese",
    testset_size: int = 10
) -> Dataset:
    """
    Generate a multilingual test dataset using RAGAS TestsetGenerator.
    
    This uses the adapt_prompts() method to translate the QA generation
    prompts to the target language before synthesizing questions.
    
    Args:
        documents: List of LangChain Document objects
        llm: Language model for generation
        embedding_model: Embedding model
        language: Target language (chinese, arabic, spanish, etc.)
        testset_size: Number of test samples to generate
    
    Returns:
        Dataset with multilingual questions and answers
    """
    # Initialize testset generator
    generator = TestsetGenerator(llm=llm, embedding_model=embedding_model)
    
    # Define query distribution
    query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=llm), 1.0)
    ]
    
    # Define transforms with LLM-based extractors
    transforms = [
        HeadlineSplitter(),
        NERExtractor(llm=llm),
        SummaryExtractor(llm=llm)
    ]
    
    # CRITICAL: Adapt prompts for target language
    # This ensures all prompts (synthesizer + transforms) are in the target language
    # See: https://github.com/vibrantlabsai/ragas/pull/2651
    
    # Adapt synthesizer prompts
    for synthesizer, _ in query_distribution:
        adapted_prompts = await synthesizer.adapt_prompts(language, llm)
        synthesizer.set_prompts(**adapted_prompts)
    
    # Adapt transform prompts
    for transform in transforms:
        if hasattr(transform, 'adapt_prompts'):
            adapted_prompts = await transform.adapt_prompts(language, llm)
            transform.set_prompts(**adapted_prompts)
    
    # Generate testset
    testset = generator.generate_with_langchain_docs(
        documents=documents,
        testset_size=testset_size,
        transforms=transforms,
        query_distribution=query_distribution
    )
    
    return testset

print("Multilingual testset generator defined!")

## Part 3: Create Sample Multilingual Documents

For this demonstration, we create sample documents in multiple languages.

In [ ]:
from langchain_core.documents import Document

# Sample multilingual documents about AI/ML
sample_docs = [
    Document(
        page_content="""Machine Learning is a subset of artificial intelligence that enables systems to learn from data. 
It includes supervised learning, unsupervised learning, and reinforcement learning. 
Deep learning uses neural networks with multiple layers to process complex patterns.
Applications include image recognition, natural language processing, and recommendation systems.""",
        metadata={"language": "english", "topic": "ml_basics"}
    ),
    Document(
        page_content="""机器学习是人工智能的一个分支，使系统能够从数据中学习。
它包括监督学习、无监督学习和强化学习。
深度学习使用多层神经网络来处理复杂的模式。
应用包括图像识别、自然语言处理和推荐系统。
在中国，机器学习被广泛应用于医疗诊断、金融风控和智能制造领域。""",
        metadata={"language": "chinese", "topic": "ml_basics"}
    ),
    Document(
        page_content="""التعلم الآلي هو فرع من الذكاء الاصطناعي يمكن الأنظمة من التعلم من البيانات.
يتضمن التعلم تحت الإشراف والتعلم غير الخاضع للإشراف والتعلم المعزز.
التعلم العميق يستخدم الشبكات العصبية متعددة الطبقات لمعالجة الأنماط المعقدة.
تشمل التطبيقات التعرف على الصور ومعالجة اللغة الطبيعية وأنظمة التوصية.""",
        metadata={"language": "arabic", "topic": "ml_basics"}
    ),
    Document(
        page_content="""Sun'iy intellektning bir turi bo'lgan mashina o'rganishi tizimlarning ma'lumotlardan o'rganishini ta'minlaydi.
U nazorat ostida o'rganish, nazoratsiz o'rganish va mustahkamlash orqali o'rganishni o'z ichiga oladi.
Chuqur o'rganish murakkab naqshlarni qayta ishlash uchun ko'p qatlamli neyron tarmoqlardan foydalanadi.
Qo'llanilish sohalari tasvirlarni tanish, tabiiy tilni qayta ishlash va tavsiya tizimlarini o'z ichiga oladi.
O'zbekistonda mashina o'rganishi tibbiyot, moliya va qishloq xo'jaligi sohalarida qo'llanilmoqda.""",
        metadata={"language": "uzbek", "topic": "ml_basics"}
    ),
]

print(f"Created {len(sample_docs)} sample documents in English, Chinese, Arabic, and Uzbek")

## Part 4: Manual Multilingual Evaluation Dataset

For reproducible evaluation, we create a structured evaluation dataset with ground truth in multiple languages.

In [ ]:
def create_multilingual_eval_dataset() -> EvaluationDataset:
    """
    Create a multilingual evaluation dataset for RAG assessment.
    
    This follows the format used in the SOAS RAG Evaluation benchmark
    (https://github.com/rajantripathi/soas-rag-evaluation)
    
    Returns:
        EvaluationDataset with samples in multiple languages
    """
    samples = [
        # English samples
        SingleTurnSample(
            user_input="What is machine learning?",
            response="Machine learning is a subset of artificial intelligence that enables systems to learn from data without being explicitly programmed.",
            reference="Machine learning is a subset of artificial intelligence that enables systems to learn from data.",
            retrieved_contexts=["Machine Learning is a subset of artificial intelligence that enables systems to learn from data."]
        ),
        SingleTurnSample(
            user_input="What are the types of machine learning?",
            response="The three main types of machine learning are supervised learning, unsupervised learning, and reinforcement learning.",
            reference="It includes supervised learning, unsupervised learning, and reinforcement learning.",
            retrieved_contexts=["It includes supervised learning, unsupervised learning, and reinforcement learning."]
        ),
        # Chinese samples
        SingleTurnSample(
            user_input="什么是机器学习？",
            response="机器学习是人工智能的一个分支，使系统能够从数据中学习，而无需明确编程。",
            reference="机器学习是人工智能的一个分支，使系统能够从数据中学习。",
            retrieved_contexts=["机器学习是人工智能的一个分支，使系统能够从数据中学习。"]
        ),
        SingleTurnSample(
            user_input="机器学习在中国有哪些应用？",
            response="在中国，机器学习被广泛应用于医疗诊断、金融风控和智能制造领域。",
            reference="在中国，机器学习被广泛应用于医疗诊断、金融风控和智能制造领域。",
            retrieved_contexts=["在中国，机器学习被广泛应用于医疗诊断、金融风控和智能制造领域。"]
        ),
        # Arabic samples
        SingleTurnSample(
            user_input="ما هو التعلم الآلي؟",
            response="التعلم الآلي هو فرع من الذكاء الاصطناعي يمكن الأنظمة من التعلم من البيانات دون برمجة صريحة.",
            reference="التعلم الآلي هو فرع من الذكاء الاصطناعي يمكن الأنظمة من التعلم من البيانات.",
            retrieved_contexts=["التعلم الآلي هو فرع من الذكاء الاصطناعي يمكن الأنظمة من التعلم من البيانات."]
        ),
        # Uzbek samples
        SingleTurnSample(
            user_input="Mashina o'rganishi nima?",
            response="Mashina o'rganishi - bu sun'iy intellektning bir turi bo'lib, tizimlarning ma'lumotlardan o'rganishini ta'minlaydi.",
            reference="Sun'iy intellektning bir turi bo'lgan mashina o'rganishi tizimlarning ma'lumotlardan o'rganishini ta'minlaydi.",
            retrieved_contexts=["Sun'iy intellektning bir turi bo'lgan mashina o'rganishi tizimlarning ma'lumotlardan o'rganishini ta'minlaydi."]
        ),
        SingleTurnSample(
            user_input="O'zbekistonda mashina o'rganishi qayerda qo'llaniladi?",
            response="O'zbekistonda mashina o'rganishi tibbiyot, moliya va qishloq xo'jaligi sohalarida qo'llanilmoqda.",
            reference="O'zbekistonda mashina o'rganishi tibbiyot, moliya va qishloq xo'jaligi sohalarida qo'llanilmoqda.",
            retrieved_contexts=["O'zbekistonda mashina o'rganishi tibbiyot, moliya va qishloq xo'jaligi sohalarida qo'llanilmoqda."]
        ),
    ]
    
    return EvaluationDataset(samples=samples)

print("Multilingual evaluation dataset function defined!")

## Part 5: Run Evaluation Across Languages

In [ ]:
# Create evaluation dataset
eval_dataset = create_multilingual_eval_dataset()

print(f"Created evaluation dataset with {len(eval_dataset.samples)} samples")

In [ ]:
# Define metrics for evaluation
metrics = [
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
    answer_correctness
]

print(f"Metrics defined: {[m.name for m in metrics]}")

In [ ]:
# Run evaluation
# Note: This requires a running LLM and may take a few minutes

async def run_evaluation():
    """Run RAGAS evaluation on the multilingual dataset."""
    results = evaluate(
        dataset=eval_dataset,
        metrics=metrics,
        llm=llm,
        embeddings=embeddings
    )
    return results

# Run the evaluation
# results = asyncio.run(run_evaluation())
# print(results)

# For demonstration, we show expected output format
print("""
Expected evaluation output:
{'faithfulness': 0.85, 'answer_relevancy': 0.92, 'context_precision': 0.88, 
 'context_recall': 0.90, 'answer_correctness': 0.87}

Note: Actual values depend on the quality of retrieved contexts and LLM responses.
""")

## Part 6: Analyze Results by Language

In [ ]:
import pandas as pd

def analyze_by_language(results_df: pd.DataFrame) -> pd.DataFrame:
    """
    Analyze evaluation results grouped by language.
    
    Args:
        results_df: DataFrame with evaluation results including language column
    
    Returns:
        DataFrame with mean scores per language
    """
    # Group by language and calculate mean scores
    language_scores = results_df.groupby('language').agg({
        'faithfulness': 'mean',
        'answer_relevancy': 'mean',
        'context_precision': 'mean',
        'context_recall': 'mean',
        'answer_correctness': 'mean'
    }).round(3)
    
    return language_scores

# Sample results for demonstration
sample_results = pd.DataFrame({
    'language': ['english', 'english', 'chinese', 'chinese', 'arabic', 'uzbek', 'uzbek'],
    'faithfulness': [0.92, 0.88, 0.85, 0.82, 0.78, 0.75, 0.80],
    'answer_relevancy': [0.95, 0.91, 0.88, 0.85, 0.82, 0.78, 0.83],
    'context_precision': [0.90, 0.88, 0.85, 0.80, 0.75, 0.72, 0.78],
    'context_recall': [0.92, 0.90, 0.87, 0.82, 0.78, 0.75, 0.80],
    'answer_correctness': [0.93, 0.89, 0.86, 0.81, 0.77, 0.74, 0.79]
})

language_analysis = analyze_by_language(sample_results)
print("Evaluation Results by Language:")
print(language_analysis.to_string())

In [ ]:
# Visualize results
import matplotlib.pyplot as plt

def plot_language_comparison(language_scores: pd.DataFrame):
    """Create a bar chart comparing scores across languages."""
    fig, ax = plt.subplots(figsize=(12, 6))
    
    x = range(len(language_scores.columns))
    width = 0.15
    
    colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12']
    
    for i, (language, scores) in enumerate(language_scores.iterrows()):
        ax.bar(
            [p + width * i for p in x],
            scores.values,
            width,
            label=language.capitalize(),
            color=colors[i % len(colors)]
        )
    
    ax.set_xlabel('Metric')
    ax.set_ylabel('Score')
    ax.set_title('RAG Evaluation Metrics by Language')
    ax.set_xticks([p + width * 1.5 for p in x])
    ax.set_xticklabels(language_scores.columns, rotation=45, ha='right')
    ax.legend()
    ax.set_ylim(0, 1)
    
    plt.tight_layout()
    plt.savefig('multilingual_rag_evaluation.png', dpi=150)
    plt.show()

plot_language_comparison(language_analysis)

## Part 7: Best Practices for Multilingual RAG

### Key Findings

1. **Use Multilingual Embeddings**: Standard OpenAI embeddings perform poorly on non-English text. Models like `paraphrase-multilingual-mpnet-base-v2` significantly improve retrieval quality.

2. **Adapt All Prompts**: When generating test data or running evaluations, ensure ALL prompts (synthesizer + transforms) are adapted to the target language. See [RAGAS PR #2651](https://github.com/vibrantlabsai/ragas/pull/2651).

3. **Language-Specific Chunking**: Different languages have different optimal chunk sizes. Chinese and Arabic benefit from character-based splitting, while English works well with word-based splitting.

4. **Ground Truth Quality**: Low-resource languages (like Uzbek) often have lower scores due to limited training data in the LLM. Manual verification of ground truth is essential.

### Common Pitfalls

- Using English-only evaluation prompts for non-English content
- Not adapting transform prompts (NER, summarization) to target language
- Ignoring tokenization differences across scripts (Latin vs. CJK vs. Arabic)
- Assuming translation quality = RAG quality

## Part 8: Advanced - Custom Multilingual Metrics

In [ ]:
from ragas.metrics._answer_relevance import AnswerRelevance
from ragas.prompt import PydanticPrompt

class MultilingualAnswerRelevance(AnswerRelevance):
    """
    Custom answer relevance metric optimized for multilingual contexts.
    
    This metric:
    1. Detects the language of the question
    2. Generates potential questions in the same language
    3. Computes similarity in a language-agnostic embedding space
    """
    
    name = "multilingual_answer_relevance"
    
    def __init__(self):
        super().__init__()
        # Override with multilingual embeddings
        self.embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
        )

print("Custom multilingual metric defined!")

## Conclusion

This notebook demonstrated a complete workflow for evaluating multilingual RAG systems using LangChain and RAGAS. Key takeaways:

1. **Multilingual embeddings are essential** for cross-lingual retrieval quality
2. **Prompt adaptation** ensures evaluation prompts match the document language
3. **Language-specific analysis** reveals performance gaps across linguistic contexts
4. **Ground truth verification** is critical for low-resource languages

### References

- [RAGAS Documentation](https://docs.ragas.io/)
- [LangChain RAG Tutorial](https://python.langchain.com/docs/tutorials/rag/)
- [SOAS RAG Evaluation Benchmark](https://github.com/rajantripathi/soas-rag-evaluation)
- [RAGAS PR #2651: adapt_all_prompts()](https://github.com/vibrantlabsai/ragas/pull/2651)
- [DeepEval Issue #2578: Multilingual Support](https://github.com/confident-ai/deepeval/issues/2578)

---

**Author**: Dr. Rajan Prasad Tripathi  
**Affiliation**: Director, AUT AI Innovation Lab  
**GitHub**: [@rajantripathi](https://github.com/rajantripathi)  
**Research Focus**: Multilingual RAG, RAG Evaluation, Healthcare AI